In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

filepath = "file:///D:/BDA 20/ABD-BDA-LabWork/datasets/0909 Lab/Weather Data in India from 1901 to 2017.csv"
months = ["JAN","FEB","MAR","APR","MAY","JUN","JUL","AUG","SEP","OCT","NOV","DEC"]

In [13]:
def create_session():
    spark = SparkSession.builder.appName('Weather').getOrCreate()
    return spark

In [14]:
def create_df(spark, filepath):
    df = spark.read.csv(filepath, header = True)
    return df

In [15]:
spark = create_session()
df = create_df(spark, filepath)
df.printSchema()

root
 |-- YEAR: string (nullable = true)
 |-- JAN: string (nullable = true)
 |-- FEB: string (nullable = true)
 |-- MAR: string (nullable = true)
 |-- APR: string (nullable = true)
 |-- MAY: string (nullable = true)
 |-- JUN: string (nullable = true)
 |-- JUL: string (nullable = true)
 |-- AUG: string (nullable = true)
 |-- SEP: string (nullable = true)
 |-- OCT: string (nullable = true)
 |-- NOV: string (nullable = true)
 |-- DEC: string (nullable = true)



In [19]:
# 1. Add column average temperature for each year 
avg_expr = sum(F.col(m) for m in months) / len(months)
df = df.withColumn("AVG", F.round(avg_expr, 2))
df.select("YEAR", "AVG").show()

+----+-----+
|YEAR|  AVG|
+----+-----+
|1901|24.22|
|1902|24.33|
|1903|23.98|
|1904|23.85|
|1905|23.93|
|1906| 24.1|
|1907|24.01|
|1908|23.95|
|1909|23.78|
|1910|23.76|
|1911|23.95|
|1912|24.11|
|1913|23.88|
|1914|24.03|
|1915|24.29|
|1916|24.08|
|1917|23.57|
|1918|23.83|
|1919|24.01|
|1920|23.91|
+----+-----+
only showing top 20 rows



In [21]:
# 2. Add row average temperature of each month 
month_avg_row = df.select(
    F.lit("AVG_OF_MONTH"),*[F.round(F.avg(m), 2).alias(m) for m in months]
)
month_avg_row.show()

+------------+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+
|AVG_OF_MONTH|  JAN|  FEB|  MAR|  APR|  MAY| JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|
+------------+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+
|AVG_OF_MONTH|18.42|20.14|23.43|26.51|28.39|28.3|27.37|26.94|26.34|24.74|21.77|19.17|
+------------+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+



In [22]:
# 3. Add columns Min and Max for each year with corresponding values 
df = df.withColumn("MIN", F.least(*[F.col(m) for m in months]))
df = df.withColumn("MAX", F.greatest(*[F.col(m) for m in months]))
df.select("YEAR", "MIN", "MAX").show(5)

+----+-----+-----+
|YEAR|  MIN|  MAX|
+----+-----+-----+
|1901|17.99| 28.6|
|1902|18.78|28.68|
|1903|18.29|28.41|
|1904|17.77|27.85|
|1905| 17.4|28.69|
+----+-----+-----+
only showing top 5 rows



In [26]:
# 4. Create dataframe called 'decade', storing average temperatures of each month in the respective 
# decade. Decade 1910 is from 1901 to 1910. 
df = df.withColumn("DECADE",(F.floor((F.col("YEAR") - 1) / 10) + 1) * 10)
 
decade = df.groupBy("DECADE").agg(
    *[F.round(F.avg(m), 2).alias(m) for m in months]
).orderBy("DECADE")
decade.show()

+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|DECADE|  JAN|  FEB|  MAR|  APR|  MAY|  JUN|  JUL|  AUG|  SEP|  OCT|  NOV|  DEC|
+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|  1910|18.15|19.38|22.83|26.28| 28.2|28.16|27.27|26.75|26.04|24.67| 21.5|18.66|
|  1920|18.32|19.62| 22.8|26.01|28.09|28.04|27.37|26.82|26.12| 24.5|21.39|18.52|
|  1930|18.22| 19.9| 23.5|26.48|28.51|28.24|27.28|26.79|26.08|24.35|21.26|18.88|
|  1940| 18.2|19.97|22.95|26.17|28.39|28.16|27.23|26.77|26.26|24.53|21.48|18.97|
|  1950|18.17|19.66|23.24|26.38|28.55|28.35|27.31|26.95|26.21|24.63|21.49|18.94|
|  1960|18.45|20.23|23.43|26.41| 28.3|28.28|27.19|26.83|26.33|24.56|21.53|19.32|
|  1970|18.13|20.14|23.47| 26.3| 28.0|28.22|27.26|26.83|26.24|24.58|21.63|18.98|
|  1980| 18.3|19.87|23.26|26.68|28.14|28.19| 27.2|26.73|26.21|24.75|21.89|19.09|
|  1990| 18.4|20.12|23.32| 26.4|28.13| 28.3|27.31|26.99|26.45|24.68|21.73|19.41|
|  2000|18.69| 20.6|23.68|26

In [27]:
# 5. Which year is the hottest year? (Based on avg temp) 
hottest = df.orderBy(F.desc("AVG")).select("YEAR", "AVG").first()
print(f"Hottest year: {hottest['YEAR']} ({hottest['AVG']})")

Hottest year: 2016 (26.45)


In [29]:
# 6. Which year is the coldest year? (Based on avg temp) 
coldest = df.orderBy(F.asc("AVG")).select("YEAR", "AVG").first()
print(f"Coldest year: {coldest['YEAR']} ({coldest['AVG']})")

Coldest year: 1917 (23.57)


In [30]:
# 7. Find out which years recorded minimum and maximum temperatures. (Not based on avg temp) 
max_row = df.orderBy(F.desc("MAX")).select("YEAR", "MAX").first()
min_row = df.orderBy(F.asc("MIN")).select("YEAR", "MIN").first()
print(f"Year with overall MAX temp: {max_row['YEAR']} ({max_row['MAX']})")
print(f"Year with overall MIN temp: {min_row['YEAR']} ({min_row['MIN']})")

Year with overall MAX temp: 1921 (30.78)
Year with overall MIN temp: 1918 (17.25)


In [34]:
# 8. How much raise is observed in min and max temperature for each month?

# First decade
first = df.filter((df["YEAR"] >= 1901) & (df["YEAR"] <= 1910))
last = df.filter((df["YEAR"] >= 2011) & (df["YEAR"] <= 2017))

print("Month | MIN rise | MAX rise")

for m in months:

    first_min = float(first.select(F.min(m)).first()[0])
    first_max = float(first.select(F.max(m)).first()[0])

    last_min = float(last.select(F.min(m)).first()[0])
    last_max = float(last.select(F.max(m)).first()[0])

    min_rise = round(last_min - first_min, 2)
    max_rise = round(last_max - first_max, 2)

    print(m, "|", min_rise, "|", max_rise)

Month | MIN rise | MAX rise
JAN | 0.85 | 1.65
FEB | 2.56 | 3.19
MAR | 1.56 | 2.51
APR | 1.26 | 2.5
MAY | 0.93 | 1.41
JUN | 0.49 | 1.01
JUL | 0.94 | 0.27
AUG | 0.82 | 0.7
SEP | 1.14 | 1.77
OCT | 0.81 | 1.08
NOV | 1.46 | 1.81
DEC | 1.45 | 2.34
